<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_45_grpc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🛰️ **Module 45 — gRPC for AI Backends (service-to-service plumbing)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 45 — gRPC for AI Backends

> When a user hits your chat UI, the call rarely goes straight to the LLM. It hops through 3-7 services: **gateway → auth → rate-limit → embedder → retriever → reranker → LLM** → back. The protocol that connects those services in production isn't REST — it's **gRPC**. It's faster, strictly typed, supports bidirectional streaming, and ships an SDK in 11 languages from a single `.proto` file.
>
> TF Serving, Triton Inference Server, Vertex AI, Vespa, Qdrant — all expose a gRPC API alongside REST. This module makes you fluent.

### What you'll cover
1. Why gRPC — the four advantages over REST
2. Protocol Buffers in 5 minutes
3. Setup — install gRPC tooling
4. Write a `.proto` file for an embedding service
5. **Code-gen** — `*_pb2.py` + `*_pb2_grpc.py` from one command
6. Implement the **server** in Python
7. Implement the **client** + run an end-to-end RPC
8. **Streaming** — server-streaming for token streams, bidi for chat
9. Deadlines, errors, interceptors, TLS — the production basics
10. When to pick gRPC vs REST vs WebSocket vs message queue


## 1 · Why gRPC over REST

| | gRPC | REST + JSON |
|---|---|---|
| Wire format | Protobuf (binary, schema'd) | JSON (text, schemaless) |
| Size on the wire | ~5× smaller | baseline |
| Speed | ~5–10× faster (HTTP/2 + binary) | baseline |
| Schema | **enforced** — you can't send the wrong field | best-effort, your problem |
| Streaming | server, client, **bidi** | mostly request-response (SSE/WebSocket for streams) |
| Language SDKs | 11 official, all from the same `.proto` | hand-rolled per language |

**Where REST still wins:** browser-direct, ad-hoc curl, public APIs you want anyone to consume. **Where gRPC wins:** anything **service-to-service** inside your own datacenter — embedding service ↔ retriever ↔ LLM router.

## 2 · Protocol Buffers in 5 minutes

Protobuf is a **schema language** + a **compact binary encoding**. You write a `.proto` file describing your messages and services, then code-gen typed bindings for every language you care about.

```protobuf
syntax = "proto3";
package ai;

message EmbedRequest {
  string text = 1;          // field number — used in the wire format
  string model = 2;
}
message EmbedResponse {
  repeated float vector = 1;   // 'repeated' = array
  int32 dim = 2;
}
service Embedder {
  rpc Embed (EmbedRequest) returns (EmbedResponse);
}
```

| Concept | What it is |
|---|---|
| `message` | a struct |
| `service` | a collection of RPC methods |
| `rpc` | one method — request type → response type |
| field number | the integer ID used on the wire (don't reuse, ever) |
| `repeated` | array / list |
| `oneof` | tagged-union (only one field set) |
| `optional` | proto3 since v3.15; allows you to distinguish unset from default |


## 3 · Setup

In [ ]:
!pip -q install grpcio grpcio-tools protobuf

In [ ]:
import os
os.makedirs("/content/proto", exist_ok=True)
os.makedirs("/content/gen",   exist_ok=True)

## 4 · Write the `.proto` file

In [ ]:
proto = '''syntax = "proto3";
package ai;

message EmbedRequest  { string text = 1; string model = 2; }
message EmbedResponse { repeated float vector = 1; int32 dim = 2; }

message ChatRequest   { string user_id = 1; string content = 2; }
message ChatToken     { string token = 1; bool done = 2; }

service Embedder {
  // unary RPC
  rpc Embed (EmbedRequest) returns (EmbedResponse);
}

service Chat {
  // server-streaming: client sends one prompt, server streams tokens back
  rpc Stream (ChatRequest) returns (stream ChatToken);
}
'''
open("/content/proto/ai.proto", "w").write(proto)
print("wrote /content/proto/ai.proto")

## 5 · Code-gen — one command builds the SDK

In [ ]:
!python -m grpc_tools.protoc \
    --proto_path=/content/proto \
    --python_out=/content/gen \
    --grpc_python_out=/content/gen \
    /content/proto/ai.proto

!ls /content/gen

You now have:
- `ai_pb2.py` — message classes (`EmbedRequest`, `EmbedResponse`, …)
- `ai_pb2_grpc.py` — service stubs (`EmbedderStub`, `EmbedderServicer`, `ChatStub`, …)

Same file generates **Go / Java / TS / Rust / etc.** clients. One schema, every language.

## 6 · Implement the server

In [ ]:
import sys; sys.path.insert(0, "/content/gen")
import ai_pb2, ai_pb2_grpc, grpc, time, hashlib
from concurrent import futures

# Stand-in embedder: deterministic 16-d "embedding" of the text
def fake_embed(text: str, dim: int = 16):
    h = hashlib.md5(text.encode()).digest()
    return [(b - 128)/128.0 for b in h[:dim]]

class EmbedderImpl(ai_pb2_grpc.EmbedderServicer):
    def Embed(self, req, ctx):
        v = fake_embed(req.text)
        return ai_pb2.EmbedResponse(vector=v, dim=len(v))

class ChatImpl(ai_pb2_grpc.ChatServicer):
    def Stream(self, req, ctx):
        for tok in ["The ","capital ","of ","France ","is ","Paris."]:
            yield ai_pb2.ChatToken(token=tok, done=False)
            time.sleep(0.05)
        yield ai_pb2.ChatToken(token="", done=True)

server = grpc.server(futures.ThreadPoolExecutor(max_workers=4))
ai_pb2_grpc.add_EmbedderServicer_to_server(EmbedderImpl(), server)
ai_pb2_grpc.add_ChatServicer_to_server(ChatImpl(), server)
server.add_insecure_port("[::]:50051")
server.start()
print("gRPC server up on :50051")

## 7 · Client — call it like a local function

In [ ]:
channel = grpc.insecure_channel("localhost:50051")
emb = ai_pb2_grpc.EmbedderStub(channel)
chat = ai_pb2_grpc.ChatStub(channel)

# unary
resp = emb.Embed(ai_pb2.EmbedRequest(text="hello world", model="m1"), timeout=2.0)
print("dim:", resp.dim, " vector[:5]:", list(resp.vector[:5]))

## 8 · Server-streaming — token streams

In [ ]:
# server streaming: iterate over the stub call → it yields tokens
for tok in chat.Stream(ai_pb2.ChatRequest(user_id="u123", content="capital of France?")):
    if tok.done: print("\n[DONE]"); break
    print(tok.token, end="", flush=True)

**Streaming flavours.**

| Pattern | proto syntax | Use case |
|---|---|---|
| Unary | `rpc M (R) returns (S)` | "embed this text" |
| Server-streaming | `returns (stream S)` | LLM token streams (this demo) |
| Client-streaming | `(stream R) returns (S)` | uploading a long doc, single summary back |
| Bidirectional | `(stream R) returns (stream S)` | live chat, ASR, multi-turn agents |


## 9 · Production basics

```python
# DEADLINES — every call should set one
resp = emb.Embed(req, timeout=0.5)         # 500 ms; raises grpc.RpcError on timeout

# ERRORS — set status + message from the server side
context.set_code(grpc.StatusCode.INVALID_ARGUMENT)
context.set_details("text must be non-empty")
return ai_pb2.EmbedResponse()

# INTERCEPTORS — tracing, auth, retries, rate-limiting at the channel/server level
class AuthInterceptor(grpc.ServerInterceptor):
    def intercept_service(self, cont, handler_call_details):
        md = dict(handler_call_details.invocation_metadata)
        if md.get("authorization") != "Bearer my-token":
            return _abort_unauthenticated()
        return cont(handler_call_details)

# TLS
creds = grpc.ssl_server_credentials([(open("server.key","rb").read(),
                                       open("server.crt","rb").read())])
server.add_secure_port("[::]:443", creds)

# REFLECTION — lets `grpcurl` discover your services without the .proto
from grpc_reflection.v1alpha import reflection
reflection.enable_server_reflection(
    [ai_pb2.DESCRIPTOR.services_by_name["Embedder"].full_name], server)
```

| Topic | Defaults to set |
|---|---|
| Deadlines | every client call sets `timeout=...` (it's a per-call deadline propagated downstream) |
| Retries | configure via service config JSON (max attempts, backoff) — server stays idempotent |
| Keepalive | tune `grpc.keepalive_time_ms` so dead idle connections drop |
| Compression | `grpc.compression.Gzip` is fine for embeddings; identity for very small messages |
| Auth | mTLS for service-to-service; token-based metadata interceptor for user-context |
| Observability | OpenTelemetry interceptor — covered in M51 |


## 10 · gRPC vs REST vs WebSocket vs MQ

| Need | Use |
|---|---|
| Public API, browser-direct, easy curl | **REST + JSON** |
| Service-to-service in your DC, low latency | **gRPC** |
| Real-time browser ↔ server (chat) | **WebSocket** or **gRPC-Web** + bidi |
| Async fan-out, durable, decoupled producers/consumers | **Kafka / NATS / Redis Streams** |
| Server-Sent Events for one-way streams to browsers | **SSE** (no client→server stream) |

**Two patterns you'll see repeatedly.**
1. **gRPC inside, REST/SSE at the edge.** Browser hits a thin Next.js / FastAPI gateway over REST/SSE; the gateway translates to gRPC and fans out internally.
2. **gRPC-Web at the edge.** Lets browsers speak gRPC directly through a small proxy (Envoy). Still rare; most teams stick with pattern 1.

### Common mistakes
- ❌ Reusing `field number`s when refactoring → silent corruption on wire.
- ❌ Forgetting **deadlines** → one slow downstream cascades into everything.
- ❌ Putting auth in the message body instead of **metadata**.
- ❌ Streaming when unary is enough — bidi is harder to operate.
- ❌ Picking gRPC for a public API where curl-friendliness matters.


## ✅ Recap
- gRPC = Protobuf schema + HTTP/2 + code-gen for 11 languages.
- One `.proto` file → `ai_pb2.py` + `ai_pb2_grpc.py` via `grpc_tools.protoc`.
- Four call shapes: unary, server-stream, client-stream, bidi.
- Always set deadlines; use interceptors for auth, tracing, retries; mTLS for service-to-service.
- Default architecture: **REST/SSE at the edge, gRPC inside.**

Cleanup:


In [ ]:
server.stop(grace=1)

Next: **M46 — Kubernetes for ML** (running these services in production).